# 🎛️ Notebook 06 — Pipeline Orchestration & Error Handling

**Goal:** Build a production-grade orchestration wrapper that runs all pipeline stages in sequence, handles errors gracefully, logs every run, and raises alerts on failure.

> **Run time:** ~8 min

## Production Pipeline Pattern
```
┌─────────────────────────────────────────────────────────┐
│  Pipeline Run (logged to pipeline_run_log)               │
│                                                          │
│  Stage 1: Bronze Ingest     ──► ✅/❌ logged             │
│  Stage 2: Silver Dims       ──► ✅/❌ logged             │
│  Stage 3: Gold Star Schema  ──► ✅/❌ logged             │
│  Stage 4: Quality Checks    ──► ✅/❌ logged             │
│                                                          │
│  On failure: stop, log error, send alert (Teams/email)   │
│  On success: update watermark, notify downstream         │
└─────────────────────────────────────────────────────────┘
```

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import traceback
import time
from datetime import datetime

# Create pipeline run log table
spark.sql("""
    CREATE TABLE IF NOT EXISTS pipeline_run_log (
        run_id          STRING,
        pipeline_name   STRING,
        stage_name      STRING,
        status          STRING,
        rows_processed  BIGINT,
        duration_secs   DOUBLE,
        error_message   STRING,
        started_at      TIMESTAMP,
        completed_at    TIMESTAMP
    ) USING DELTA
""")

print('Pipeline run log table ready')

## Step 1 — Pipeline Framework Functions

In [ ]:
import uuid

PIPELINE_NAME = 'banking_daily_pipeline'
RUN_ID        = str(uuid.uuid4())[:8]  # short unique run ID

def log_stage(stage_name, status, rows=0, duration=0.0, error=None):
    """Write a single stage result to the pipeline run log."""
    log_row = spark.createDataFrame([(
        RUN_ID, PIPELINE_NAME, stage_name, status,
        rows, duration, error or '',
        datetime.utcnow(), datetime.utcnow()
    )], ['run_id','pipeline_name','stage_name','status',
         'rows_processed','duration_secs','error_message','started_at','completed_at'])
    log_row.write.format('delta').mode('append').saveAsTable('pipeline_run_log')


def run_stage(stage_name, func):
    """Run a pipeline stage, catch errors, and log the result."""
    print(f'  ⏳ {stage_name}...', end='', flush=True)
    start = time.time()
    try:
        rows = func()
        duration = round(time.time() - start, 2)
        log_stage(stage_name, 'SUCCESS', rows=rows or 0, duration=duration)
        print(f' ✅  ({duration}s, {rows or 0} rows)')
        return True
    except Exception as e:
        duration = round(time.time() - start, 2)
        err_msg  = str(e)[:500]
        log_stage(stage_name, 'FAILED', duration=duration, error=err_msg)
        print(f' ❌  ({duration}s)')
        print(f'     Error: {err_msg[:120]}')
        return False

print(f'Pipeline framework ready. Run ID: {RUN_ID}')

## Step 2 — Define Pipeline Stages

In [ ]:
def stage_validate_sources():
    """Check all source tables exist and have data."""
    required = ['bronze_customers','bronze_accounts','bronze_transactions','bronze_loans',
                'dim_customer','dim_account','fact_transactions','fact_loans']
    for t in required:
        count = spark.sql(f'SELECT COUNT(*) AS n FROM {t}').collect()[0]['n']
        if count == 0:
            raise ValueError(f'Table {t} is empty!')
    return len(required)


def stage_quality_check():
    """Run quality checks on fact_transactions."""
    results = spark.sql("""
        SELECT
            COUNT(*) AS total,
            SUM(CASE WHEN Amount <= 0 THEN 1 ELSE 0 END)   AS bad_amounts,
            SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customers,
            SUM(CASE WHEN AccountID IS NULL THEN 1 ELSE 0 END)  AS null_accounts
        FROM fact_transactions
    """).collect()[0]
    if results['bad_amounts'] > 0 or results['null_customers'] > 0:
        raise ValueError(f'Quality check failed: bad_amounts={results["bad_amounts"]}, null_customers={results["null_customers"]}')
    return results['total']


def stage_refresh_gold_branch():
    """Refresh gold_branch_performance."""
    df = spark.sql("""
        SELECT BranchID, BranchName, Region,
               COUNT(LoanID)                                    AS TotalLoans,
               ROUND(SUM(LoanAmount), 0)                        AS TotalLoanVolume,
               ROUND(AVG(LoanAmount), 0)                        AS AvgLoanSize,
               ROUND(AVG(InterestRate), 2)                      AS AvgInterestRate,
               SUM(IsDefault)                                   AS TotalDefaults,
               ROUND(SUM(IsDefault)*100.0/COUNT(LoanID), 1)    AS DefaultRate_Pct,
               ROUND(SUM(OutstandingBalance), 0)                AS TotalOutstanding
        FROM fact_loans
        GROUP BY BranchID, BranchName, Region
    """)
    df.write.format('delta').mode('overwrite').saveAsTable('gold_branch_performance')
    return df.count()


def stage_update_watermark():
    """Update pipeline watermark to current timestamp."""
    spark.sql(f"""
        MERGE INTO pipeline_watermarks t
        USING (SELECT '{PIPELINE_NAME}' AS pipeline_name) s
        ON t.pipeline_name = s.pipeline_name
        WHEN MATCHED     THEN UPDATE SET last_watermark = date_format(current_timestamp(), 'yyyy-MM-dd'),
                                         updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT VALUES (s.pipeline_name, date_format(current_timestamp(), 'yyyy-MM-dd'), 0, current_timestamp())
    """)
    return 1

print('Pipeline stages defined')

## Step 3 — Run the Full Pipeline

In [ ]:
print('=' * 55)
print(f'  PIPELINE: {PIPELINE_NAME}')
print(f'  RUN ID:   {RUN_ID}')
print(f'  STARTED:  {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")} UTC')
print('=' * 55)

stages = [
    ('Validate Source Tables',  stage_validate_sources),
    ('Quality Check',           stage_quality_check),
    ('Refresh Gold: Branch',    stage_refresh_gold_branch),
    ('Update Watermark',        stage_update_watermark),
]

all_passed = True
for stage_name, func in stages:
    success = run_stage(stage_name, func)
    if not success:
        all_passed = False
        print(f'\n❌ Pipeline ABORTED at stage: {stage_name}')
        print('   Check pipeline_run_log for details.')
        break

print('\n' + '=' * 55)
if all_passed:
    print(f'  ✅ PIPELINE COMPLETED SUCCESSFULLY')
else:
    print(f'  ❌ PIPELINE FAILED — see run log')
print(f'  ENDED: {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")} UTC')
print('=' * 55)

## Step 4 — View Pipeline Run Log

In [ ]:
%%sql
SELECT run_id, stage_name, status, rows_processed,
       ROUND(duration_secs, 2) AS duration_secs,
       error_message
FROM pipeline_run_log
WHERE run_id = (SELECT run_id FROM pipeline_run_log ORDER BY started_at DESC LIMIT 1)
ORDER BY started_at

## Step 5 — Historical Run Summary

In [ ]:
%%sql
-- All pipeline runs summary
SELECT
    run_id,
    pipeline_name,
    COUNT(*)                                           AS TotalStages,
    SUM(CASE WHEN status='SUCCESS' THEN 1 ELSE 0 END)  AS Passed,
    SUM(CASE WHEN status='FAILED'  THEN 1 ELSE 0 END)  AS Failed,
    ROUND(SUM(duration_secs), 1)                        AS TotalDuration_Secs,
    MIN(started_at)                                     AS RunStarted
FROM pipeline_run_log
GROUP BY run_id, pipeline_name
ORDER BY RunStarted DESC